In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM
from adapters import Adapter

# Metas

In [33]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [34]:
priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [35]:
context_manager = ContextManager()

# Model Family

In [46]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=priors,
    intrinsic_params=intrinsic_params,
)

In [49]:
samples = model_family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500
)

(180,) (180,)
(180,) (180,)
(180,) (180,)


In [48]:
samples["param_masks"].shape

(3, 180)

In [44]:
samples["param_matrices"].shape

(3, 180)

In [45]:
samples["regressor_masks"].shape

(3, 30)

In [50]:
samples

{'model_names': ['DDM', 'DDM', 'DDM'],
 'design_configs': [{'u_0': ['v', 'tau'],
   'u_1': [],
   'u_2': ['decay'],
   'u_3': ['v', 'a', 's_v'],
   'u_4': ['a'],
   'u_5': ['v', 's_v', 'decay'],
   'u_6': ['v', 'a', 'tau', 'decay'],
   'u_7': ['a', 'tau', 'decay'],
   'u_8': ['a', 's_v', 'decay'],
   'u_9': ['v', 's_v', 'decay']},
  {'u_0': ['v', 's_v'],
   'u_1': ['s_v'],
   'u_2': ['v', 'a', 'tau', 's_v', 'decay'],
   'u_3': ['tau', 's_v', 'decay'],
   'u_4': ['v', 'a', 's_v'],
   'u_5': ['v', 'tau'],
   'u_6': ['tau'],
   'u_7': ['v', 'a', 's_v', 'decay'],
   'u_8': ['a', 's_v', 'decay'],
   'u_9': ['v', 'tau']},
  {'u_0': ['a', 'tau', 's_v'],
   'u_1': ['v', 'tau', 'decay'],
   'u_2': ['s_v'],
   'u_3': ['v', 'a', 'tau', 's_v'],
   'u_4': ['v', 'a', 'tau', 's_v', 'decay'],
   'u_5': ['v', 'tau', 'decay'],
   'u_6': ['a', 's_v', 'decay'],
   'u_7': ['v', 'a', 'tau'],
   'u_8': ['a', 'tau', 's_v'],
   'u_9': ['v', 'tau', 'decay']}],
 'design_matrices': array([[[0.        , 0.        

# Adapter

In [24]:
adapter = Adapter()

In [25]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

In [26]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

10 2798 30


In [27]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [28]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)